# Preparation of preliminary ESCO-KldB Crosswalk
Felix Zaussinger | XX.YY.ZZZZ

## Core Analysis Goal(s)
1.
2.
3.

## Key Insight(s)
1.
2.
3.

In [3]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils, plotting_utils

# import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("talk")
sns.set(rc={"figure.figsize": (6, 3.0)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

In [4]:
def print_uniques_per_col(df):
    for col in df.columns:
        print(col, " : ", df[col].unique().shape[0])

#### Prepare ESCO - KldB crosswalk

Two steps:
1. ESCO - Berufenet DKZ ID (3-6 digits)
2. DKZ ID - KldB 2010 code (5 digits)

In [5]:
from src.data.framework import Crosswalks

crosswalks = Crosswalks()
unique_occs = crosswalks.esco_de_kldb2010.Classification_2_ID.dropna().unique()
len_of_occ_codes = pd.Series(unique_occs).astype(int).astype(str).str.len()
np.unique(len_of_occ_codes, return_counts=True)

(array([3, 4, 5, 6], dtype=int64), array([ 78, 993, 869, 140], dtype=int64))

In [6]:
print_uniques_per_col(crosswalks.esco_de_kldb2010)

Classification_1_URI  :  2942
Classification_1_PrefLabel  :  2941
Classification_1_URL  :  2942
Classification_2_ID  :  2081
Classification_2_PrefLabel  :  2081
Classification_2_URL  :  2081
Mapping_relation  :  5


< 2900 occs in this crosswalk compared to 2000 in IAB version, therefore using this one but combined with DKZ-kldb2010 mapping from IAB

In [7]:
crosswalks.esco_de_kldb2010.Classification_1_URI.unique()

array([nan,
       'http://data.europa.eu/esco/occupation/7696b15a-c52e-4ac5-b38d-9eeb010c5fa3',
       'http://data.europa.eu/esco/occupation/262f21a3-ae78-46f4-a5f9-5a1f502caa90',
       ...,
       'http://data.europa.eu/esco/occupation/0c448a27-10ec-43ba-b880-d9938bade424',
       'http://data.europa.eu/esco/occupation/4d27152a-a8ee-4f5a-9f93-a2fb4fb2b2e3',
       'http://data.europa.eu/esco/occupation/ceda6443-e1dd-4890-b018-42318e3abac2'],
      dtype=object)

In [8]:
# version 1
esco_kldb_iab = pd.read_excel(
    r"T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\data\raw\crosswalks\ESCO-Zuordnungen.xlsx",
    sheet_name="ESCO-Systematik zu Beruf",
)

dkz_to_kldb = esco_kldb_iab[["DKZ-ID", "Kldb2010"]]

# version 2: https://download-portal.arbeitsagentur.de/files/personList.do?sortierfeld=&doNext=listAnzeigen&doNext=list&seite=1&anzahlProSeite=15
dkz_to_kldb_v2 = pd.read_excel(
    r"T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\data\raw\crosswalks\DKZ_Berufe_Zuordnung_Berufsgattung.xlsx",
    sheet_name="Berufe_Berufsgattung KldB 2010",
    header=3,
    usecols=["DKZ-ID Beruf", "Codenr. KldB 2010 Berufsgattung Numerisch"],
)

In [9]:
# Version 1
print_uniques_per_col(dkz_to_kldb)

# Version 2
print_uniques_per_col(dkz_to_kldb_v2)

DKZ-ID  :  4822
Kldb2010  :  1203
DKZ-ID Beruf  :  31245
Codenr. KldB 2010 Berufsgattung Numerisch  :  1300


In [10]:
cw_kldb = crosswalks.esco_de_kldb2010
cw_kldb_cleaned = cw_kldb.dropna(subset=["Classification_1_URI", "Classification_2_ID"])
cw_kldb_cleaned.Classification_2_ID = cw_kldb_cleaned.Classification_2_ID.astype(int)

print_uniques_per_col(cw_kldb_cleaned)

Classification_1_URI  :  2792
Classification_1_PrefLabel  :  2791
Classification_1_URL  :  2792
Classification_2_ID  :  2079
Classification_2_PrefLabel  :  2079
Classification_2_URL  :  2079
Mapping_relation  :  4


C:\Users\fzaussinger\AppData\Local\Temp\ipykernel_7844\3890078982.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cw_kldb_cleaned.Classification_2_ID = cw_kldb_cleaned.Classification_2_ID.astype(int)


Version 1

In [22]:
cw_kldb_merged = pd.merge(
    cw_kldb_cleaned,
    dkz_to_kldb,
    left_on="Classification_2_ID",
    right_on="DKZ-ID",
    how="left",
).drop_duplicates()
cw_kldb_merged = cw_kldb_merged.dropna(subset=["Kldb2010"])
cw_kldb_merged[["DKZ-ID", "Kldb2010"]] = cw_kldb_merged[["DKZ-ID", "Kldb2010"]].astype(
    int
)
cw_kldb_merged = cw_kldb_merged.rename(columns={"Kldb2010": "kldb2010_code"})

print_uniques_per_col(cw_kldb_merged)

Classification_1_URI  :  2778
Classification_1_PrefLabel  :  2778
Classification_1_URL  :  2778
Classification_2_ID  :  2055
Classification_2_PrefLabel  :  2055
Classification_2_URL  :  2055
Mapping_relation  :  4
DKZ-ID  :  2055
kldb2010_code  :  920


Version 2

In [23]:
cw_kldb_merged_v2 = pd.merge(
    cw_kldb_cleaned,
    dkz_to_kldb_v2,
    left_on="Classification_2_ID",
    right_on="DKZ-ID Beruf",
    how="left",
).drop_duplicates()
cw_kldb_merged_v2 = cw_kldb_merged_v2.dropna(
    subset=["Codenr. KldB 2010 Berufsgattung Numerisch"]
)
cw_kldb_merged_v2[
    ["DKZ-ID Beruf", "Codenr. KldB 2010 Berufsgattung Numerisch"]
] = cw_kldb_merged_v2[
    ["DKZ-ID Beruf", "Codenr. KldB 2010 Berufsgattung Numerisch"]
].astype(
    int
)
cw_kldb_merged_v2 = cw_kldb_merged_v2.rename(
    columns={
        "Codenr. KldB 2010 Berufsgattung Numerisch": "kldb2010_code",
        "DKZ-ID Beruf": "DKZ-ID",
    }
)

print_uniques_per_col(cw_kldb_merged_v2)

Classification_1_URI  :  2792
Classification_1_PrefLabel  :  2791
Classification_1_URL  :  2792
Classification_2_ID  :  2079
Classification_2_PrefLabel  :  2079
Classification_2_URL  :  2079
Mapping_relation  :  4
DKZ-ID  :  2079
kldb2010_code  :  920


In [24]:
cw_kldb_merged_v2

,Classification_1_URI,Classification_1_PrefLabel,Classification_1_URL,Classification_2_ID,Classification_2_PrefLabel,Classification_2_URL,Mapping_relation,DKZ-ID,kldb2010_code
0,http://data.europa.eu/esco/occupation/7696b15a...,Artillerieoffizier/Artillerieoffizierin,http://data.europa.eu/esco/occupation/7696b15a...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,15341,1104
1,http://data.europa.eu/esco/occupation/262f21a3...,Marineoffizier/Marineoffizierin,http://data.europa.eu/esco/occupation/262f21a3...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,15341,1104
2,http://data.europa.eu/esco/occupation/c6a26e11...,Offizier für die Streitkräfte/Offizierin für d...,http://data.europa.eu/esco/occupation/c6a26e11...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:exactMatch,15341,1104
3,http://data.europa.eu/esco/occupation/e44b2459...,Kommandeur eines Geschwaders oder einer Kompan...,http://data.europa.eu/esco/occupation/e44b2459...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,15341,1104
4,http://data.europa.eu/esco/occupation/f2cc5978...,Luftwaffenoffizier/Luftwaffenoffizierin,http://data.europa.eu/esco/occupation/f2cc5978...,15341,Offizier - Truppendienst,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,15341,1104
...,...,...,...,...,...,...,...,...,...
4053,http://data.europa.eu/esco/occupation/4d27152a...,Kindergartenhelfer/Kindergartenhelferin,http://data.europa.eu/esco/occupation/4d27152a...,13941,Sozialassistent/in,https://berufenet.arbeitsagentur.de/berufenet/...,skos:closeMatch,13941,83142
4054,http://data.europa.eu/esco/occupation/ceda6443...,Sophrologe/Sophrologin,http://data.europa.eu/esco/occupation/ceda6443...,9582,Yogalehrer/in,https://berufenet.arbeitsagentur.de/berufenet/...,skos:closeMatch,9582,84553
4055,http://data.europa.eu/esco/occupation/92df4ee3...,Telefonberater Krisen und Notlagen/Telefonbera...,http://data.europa.eu/esco/occupation/92df4ee3...,58699,Theologe/Theologin - evangelisch,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,58699,83314
4056,http://data.europa.eu/esco/occupation/92df4ee3...,Telefonberater Krisen und Notlagen/Telefonbera...,http://data.europa.eu/esco/occupation/92df4ee3...,58749,Theologe/Theologin - katholisch,https://berufenet.arbeitsagentur.de/berufenet/...,skos:broadMatch,58749,83314


Kldb 2010 names

In [13]:
kldb_names = pd.read_excel(
    r"T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\rcode\B_Metadaten\Kldb2010-Englisch.xls",
    sheet_name="OccupationMetadata",
    usecols=["kldb2010_code", "kldb2010_name_de", "kldb2010_name_en"],
)
kldb_names

,kldb2010_code,kldb2010_name_de,kldb2010_name_en
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
1,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
2,11103,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
3,11104,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...
4,11113,Berufe in der Landtechnik - komplexe Spezialis...,Technical occupations in farming-complex tasks
...,...,...,...
1281,94794,Führungskräfte im Museum,Managers in museum
1282,1104,Offiziere,Commissioned officers
1283,1203,Unteroffiziere mit Portepee,Senior non-commissioned officers and higher
1284,1302,Unteroffiziere ohne Portepee,Junior non-commissioned officers


Combine all data and save

In [45]:
cw_kldb_merged_final = pd.merge(
    cw_kldb_merged_v2, kldb_names, on="kldb2010_code", how="left"
)
cw_kldb_merged_final.to_csv(
    os.path.join(
        useful_paths.data_processed, "crosswalks", "crosswalk_esco_dkz_kldb2010.csv"
    )
)
cw_kldb_merged_final.to_pickle(
    os.path.join(
        useful_paths.data_processed, "crosswalks", "crosswalk_esco_dkz_kldb2010.pkl"
    )
)

In [46]:
print_uniques_per_col(cw_kldb_merged_final)

Classification_1_URI  :  2792
Classification_1_PrefLabel  :  2791
Classification_1_URL  :  2792
Classification_2_ID  :  2079
Classification_2_PrefLabel  :  2079
Classification_2_URL  :  2079
Mapping_relation  :  4
DKZ-ID  :  2079
kldb2010_code  :  920
kldb2010_name_de  :  914
kldb2010_name_en  :  914


In [47]:
print_uniques_per_col(cw_kldb_merged_final.dropna(subset=["kldb2010_code"]))

Classification_1_URI  :  2792
Classification_1_PrefLabel  :  2791
Classification_1_URL  :  2792
Classification_2_ID  :  2079
Classification_2_PrefLabel  :  2079
Classification_2_URL  :  2079
Mapping_relation  :  4
DKZ-ID  :  2079
kldb2010_code  :  920
kldb2010_name_de  :  914
kldb2010_name_en  :  914
